# Datathon PosTech - Associação Passos Mágicos 🪄

## 1. Resumo do Projeto

### O Contexto
A **Associação Passos Mágicos** atua há mais de 30 anos na transformação da vida de crianças e jovens de baixa renda do município de Embu-Guaçu, levando melhores oportunidades através da educação. Desde 2016, a ONG atua como um projeto social e educacional focado em educação de qualidade, auxílio psicológico/psicopedagógico e protagonismo juvenil.

### O Objetivo
Este projeto tem como meta principal utilizar técnicas de *Data Analytics* e *Machine Learning* aplicadas aos dados da Pesquisa Extensiva do Desenvolvimento Educacional (PEDE) dos anos de 2022, 2023 e 2024. Nossos objetivos específicos são:
1. Responder a **11 dores de negócio** da instituição através da exploração visual dos dados.
2. Desenvolver um **modelo preditivo** capaz de identificar precocemente alunos que correm o risco de queda de desempenho ou aumento de defasagem.

### Entendendo a Avaliação dos Alunos (INDE)
Para compreender as análises e o modelo a seguir, é fundamental entender que o grande consolidador do desempenho dos alunos na ONG é o **INDE (Índice de Desenvolvimento Educacional)**. Ele avalia as dimensões Acadêmica, Psicossocial e Psicopedagógica, classificando os alunos em quatro "Pedras" evolutivas: **Quartzo, Ágata, Ametista e Topázio**.

O INDE é formado por 7 indicadores chave:
* **IAN (Adequação de Nível):** Mede a defasagem (Fase Efetiva - Fase Ideal). É a métrica mais importante para o alerta de risco.
* **IDA (Desempenho Acadêmico):** A média das provas internas de Matemática, Português e Inglês.
* **IEG (Engajamento):** Entrega de lições de casa e voluntariado.
* **IAA (Autoavaliação):** Como o aluno avalia a si mesmo, seus estudos e emoções.
* **Indicadores de Conselho (IPS, IPP e IPV):** Avaliações de psicólogos e pedagogos sobre o lado psicossocial, cognitivo e o "Ponto de Virada" do aluno.

In [ ]:
# Importando as bibliotecas essenciais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

In [ ]:
file_path = '/content/PEDE_PASSOS_DATASET_FIAP.xlsx'

# Lendo as abas específicas de cada ano do arquivo Excel
print("Carregando as bases anuais...")
df_2022 = pd.read_excel(file_path, sheet_name='PEDE2022')
df_2023 = pd.read_excel(file_path, sheet_name='PEDE2023')
df_2024 = pd.read_excel(file_path, sheet_name='PEDE2024')

Carregando as bases anuais...


In [ ]:
print(f"Tamanho da base no ano de 2022: {len(df_2022)} registros")
print(f"Tamanho da base no ano de 2023: {len(df_2023)} registros")
print(f"Tamanho da base no ano de 2024: {len(df_2024)} registros")

Tamanho da base no ano de 2022: 860 registros
Tamanho da base no ano de 2023: 1014 registros
Tamanho da base no ano de 2024: 1156 registros


In [ ]:
# Lista de todas as colunas qualitativas, nominais ou de rankings que não serão usadas no modelo
colunas_para_remover = [
    # Identificadores Pessoais
    'Nome', 'Nome Anonimizado', 'Ano nasc',

    # Avaliadores Nominais
    'Nº Av', 'Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4',
    'Avaliador5', 'Avaliador6', 'Avaliador7', 'Turma', 'Escola', 'Ativo/ Inativo', 'Ativo/ Inativo.1',

    # RDados nulos em 2023 e 2024
    'Rec Av1', 'Rec Av2', 'Rec Av3', 'Rec Av4', 'Rec Psicologia', 'Indicado', 'Atingiu PV', 'Destaque IEG', 'Destaque IDA', 'Destaque IPV', 'Cf', 'Cg', 'Ct'
]

In [ ]:
df_2022 = df_2022.drop(columns=colunas_para_remover, errors='ignore')
df_2023 = df_2023.drop(columns=colunas_para_remover, errors='ignore')
df_2024 = df_2024.drop(columns=colunas_para_remover, errors='ignore')

In [ ]:
df_2022.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 860 entries, 0 to 859
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RA                     860 non-null    object 
 1   Fase                   860 non-null    int64  
 2   Idade 22               860 non-null    int64  
 3   Gênero                 860 non-null    object 
 4   Ano ingresso           860 non-null    int64  
 5   Instituição de ensino  860 non-null    object 
 6   Pedra 20               323 non-null    object 
 7   Pedra 21               462 non-null    object 
 8   Pedra 22               860 non-null    object 
 9   INDE 22                860 non-null    float64
 10  IAA                    860 non-null    float64
 11  IEG                    860 non-null    float64
 12  IPS                    860 non-null    float64
 13  IDA                    860 non-null    float64
 14  Matem                  858 non-null    float64
 15  Portug

In [ ]:
df_2023.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RA                     1014 non-null   object 
 1   Fase                   1014 non-null   object 
 2   INDE 2023              931 non-null    float64
 3   Pedra 2023             931 non-null    object 
 4   Data de Nasc           1014 non-null   object 
 5   Idade                  1014 non-null   object 
 6   Gênero                 1014 non-null   object 
 7   Ano ingresso           1014 non-null   int64  
 8   Instituição de ensino  1014 non-null   object 
 9   Pedra 20               240 non-null    object 
 10  Pedra 21               335 non-null    object 
 11  Pedra 22               600 non-null    object 
 12  Pedra 23               0 non-null      float64
 13  INDE 22                600 non-null    float64
 14  INDE 23                0 non-null      float64
 15  IAA 

In [ ]:
df_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1156 entries, 0 to 1155
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   RA                     1156 non-null   object        
 1   Fase                   1156 non-null   object        
 2   INDE 2024              1092 non-null   object        
 3   Pedra 2024             1092 non-null   object        
 4   Data de Nasc           1156 non-null   datetime64[ns]
 5   Idade                  1156 non-null   int64         
 6   Gênero                 1156 non-null   object        
 7   Ano ingresso           1156 non-null   int64         
 8   Instituição de ensino  1155 non-null   object        
 9   Pedra 20               191 non-null    object        
 10  Pedra 21               264 non-null    object        
 11  Pedra 22               472 non-null    object        
 12  Pedra 23               690 non-null    object        
 13  IND

In [ ]:
# 1. Tratamento da coluna 'Fase' na base de 2023
# Converte para texto, substitui 'ALFA' por '0' e remove a palavra 'FASE '
df_2023['Fase'] = df_2023['Fase'].astype(str).str.upper()
df_2023['Fase'] = df_2023['Fase'].str.replace('ALFA', '0', regex=False)
df_2023['Fase'] = df_2023['Fase'].str.replace('FASE ', '', regex=False)

In [ ]:
# 2. Tratamento da coluna 'Fase' na base de 2024
# Converte para texto, substitui 'ALFA' por '0' e extrai apenas o primeiro número encontrado
df_2024['Fase'] = df_2024['Fase'].astype(str).str.upper()
df_2024['Fase'] = df_2024['Fase'].str.replace('ALFA', '0', regex=False)
df_2024['Fase'] = df_2024['Fase'].str.extract(r'(\d+)') # Extrai apenas os dígitos

In [ ]:
# Tratamento de exclusão e recálculo de Idade.

# 1. Excluindo as colunas solicitadas na PEDE 2023 e PEDE 2024
df_2023 = df_2023.drop(columns=['Pedra 23', 'Idade'], errors='ignore')
df_2024 = df_2024.drop(columns=['Data de Nasc'], errors='ignore')

# 2. Recalculando a Idade para a PEDE 2023
# Primeiro, garantimos que a coluna 'Data de Nasc' seja interpretada como data pelo Pandas
df_2023['Data de Nasc'] = pd.to_datetime(df_2023['Data de Nasc'], errors='coerce')

# Em seguida, pegamos o ano de 2023 e subtraímos pelo ano de nascimento extraído da data
df_2023['Idade'] = 2023 - df_2023['Data de Nasc'].dt.year

# (Opcional) Podemos remover a 'Data de Nasc' de 2023 agora que a Idade foi calculada
df_2023 = df_2023.drop(columns=['Data de Nasc'], errors='ignore')

print("Processo concluído com sucesso!")
print(f"Colunas do df_2023 ajustadas. Exemplo das novas idades calculadas: \n{df_2023['Idade'].head()}")

Processo concluído com sucesso!
Colunas do df_2023 ajustadas. Exemplo das novas idades calculadas: 
0    8
1    9
2    7
3    8
4    9
Name: Idade, dtype: int32


In [ ]:
df_2023['Fase'] = pd.to_numeric(df_2023['Fase'], errors='coerce')
df_2024['Fase'] = pd.to_numeric(df_2024['Fase'], errors='coerce')

In [ ]:
print("\nValores únicos de Fase em 2022:", sorted(df_2022['Fase'].dropna().unique()))
print("Valores únicos de Fase em 2023:", sorted(df_2023['Fase'].dropna().unique()))
print("Valores únicos de Fase em 2024:", sorted(df_2024['Fase'].dropna().unique()))


Valores únicos de Fase em 2022: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Valores únicos de Fase em 2023: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
Valores únicos de Fase em 2024: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


In [ ]:
# 1. Ajustes na PEDE 2022
df_2022 = df_2022.rename(columns={
    'Fase ideal':'Fase Ideal',
    'Matem':'Mat',
    'Portug':'Por',
    'Inglês':'Ing',
    'Defas': 'Defasagem',
    'Idade 22': 'Idade',
    'Pedra 20': 'Pedra -2',
    'Pedra 21': 'Pedra -1',
    'Pedra 22': 'Pedra',
    'INDE 22': 'INDE'
})

# 2. Ajustes na PEDE 2023
df_2023 = df_2023.rename(columns={
    'Pedra 20': 'Pedra -3',
    'Pedra 21': 'Pedra -2',
    'Pedra 22': 'Pedra -1',
    'Pedra 2023': 'Pedra',
    'INDE 22': 'INDE - 1',
    'INDE 2023': 'INDE'
})

# Excluindo a coluna INDE 23 conforme solicitado
df_2023 = df_2023.drop(columns=['INDE 23'], errors='ignore')
df_2023 = df_2023.drop(columns=['Destaque IPV.1'], errors='ignore')


# 3. Ajustes na PEDE 2024
df_2024 = df_2024.rename(columns={
    'Pedra 20': 'Pedra -4',
    'Pedra 21': 'Pedra -3',
    'Pedra 22': 'Pedra -2',
    'Pedra 23': 'Pedra -1',
    'INDE 23': 'INDE - 1',
    'INDE 22': 'INDE - 2',
    'INDE 2024': 'INDE',
    'Pedra 2024': 'Pedra'
})

In [ ]:
# 1. Criando a nova coluna 'Ano' em cada DataFrame e a IPP em 2022
# De acordo com a metodologia oficial da Associação Passos Mágicos, o INDE para os alunos do ensino básico (Fases 0 a 7) é
# uma média ponderada exata dos 7 indicadores.
# A fórmula é: INDE = (IAN × 0,1) + (IDA × 0,2) + (IEG × 0,2) + (IAA × 0,1) + (IPS × 0,1) + (IPP × 0,1) + (IPV × 0,2)

df_2022['IPP'] =  (df_2022['INDE'] - (df_2022['IAN'] * 0.1 +
     df_2022['IDA'] * 0.2 +
     df_2022['IEG'] * 0.2 +
     df_2022['IAA'] * 0.1 +
     df_2022['IPS'] * 0.1 +
     df_2022['IPV'] * 0.2))/0.1

df_2022['Ano'] = 2022
df_2023['Ano'] = 2023
df_2024['Ano'] = 2024

In [ ]:
def reorder_columns_dynamically(df, desired_order):
    # Get existing columns in the desired order
    existing_in_order = [col for col in desired_order if col in df.columns]
    # Get columns not in the desired order, maintaining their original relative order
    other_columns = [col for col in df.columns if col not in desired_order]
    # Combine them: desired first, then others
    return df[existing_in_order + other_columns]

ordem_desejada = [
    #'RA',
    'Ano',
    'Idade',
    'Gênero',
    'Ano ingresso',
    'Instituição de ensino',
    'INDE',
    'Fase',
    'Fase Ideal',
    'Defasagem',
    'IAA',
    'IEG',
    'IPS',
    'IPP',
    'IDA',
    'Mat',
    'Por',
    'Ing',
    'IPV',
    'IAN',
    'Pedra',
    'Pedra -1',
    'Pedra -2'
]


# Aplicando a nova ordem a cada DataFrame
df_2022 = reorder_columns_dynamically(df_2022, ordem_desejada)
df_2023 = reorder_columns_dynamically(df_2023, ordem_desejada)
df_2024 = reorder_columns_dynamically(df_2024, ordem_desejada)

In [ ]:
df_2022.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 860 entries, 0 to 859
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Ano                    860 non-null    int64  
 1   Idade                  860 non-null    int64  
 2   Gênero                 860 non-null    object 
 3   Ano ingresso           860 non-null    int64  
 4   Instituição de ensino  860 non-null    object 
 5   INDE                   860 non-null    float64
 6   Fase                   860 non-null    int64  
 7   Fase Ideal             860 non-null    object 
 8   Defasagem              860 non-null    int64  
 9   IAA                    860 non-null    float64
 10  IEG                    860 non-null    float64
 11  IPS                    860 non-null    float64
 12  IPP                    860 non-null    float64
 13  IDA                    860 non-null    float64
 14  Mat                    858 non-null    float64
 15  Por   

In [ ]:
df_2023.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Ano                    1014 non-null   int64  
 1   Idade                  1014 non-null   int32  
 2   Gênero                 1014 non-null   object 
 3   Ano ingresso           1014 non-null   int64  
 4   Instituição de ensino  1014 non-null   object 
 5   INDE                   931 non-null    float64
 6   Fase                   1014 non-null   int64  
 7   Fase Ideal             1014 non-null   object 
 8   Defasagem              1014 non-null   int64  
 9   IAA                    951 non-null    float64
 10  IEG                    938 non-null    float64
 11  IPS                    945 non-null    float64
 12  IPP                    938 non-null    float64
 13  IDA                    937 non-null    float64
 14  Mat                    937 non-null    float64
 15  Por 

In [ ]:
df_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1156 entries, 0 to 1155
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Ano                    1156 non-null   int64  
 1   Idade                  1156 non-null   int64  
 2   Gênero                 1156 non-null   object 
 3   Ano ingresso           1156 non-null   int64  
 4   Instituição de ensino  1155 non-null   object 
 5   INDE                   1092 non-null   object 
 6   Fase                   1156 non-null   int64  
 7   Fase Ideal             1156 non-null   object 
 8   Defasagem              1156 non-null   int64  
 9   IAA                    1054 non-null   float64
 10  IEG                    1156 non-null   float64
 11  IPS                    1054 non-null   float64
 12  IPP                    1054 non-null   float64
 13  IDA                    1055 non-null   float64
 14  Mat                    1051 non-null   float64
 15  Por 

In [ ]:
df_2024.head()

,Ano,Idade,Gênero,Ano ingresso,Instituição de ensino,INDE,Fase,Fase Ideal,Defasagem,IAA,...,IPV,IAN,Pedra,Pedra -1,Pedra -2,RA,Pedra -4,Pedra -3,INDE - 2,INDE - 1
0,2024,8,Masculino,2024,Pública,7.611367,0,ALFA (1° e 2° ano),0,10.002,...,5.446667,10.0,Ametista,NaN,NaN,RA-1275,NaN,NaN,NaN,NaN
1,2024,8,Feminino,2024,Pública,8.002867,0,ALFA (1° e 2° ano),0,10.002,...,7.050000,10.0,Topázio,NaN,NaN,RA-1276,NaN,NaN,NaN,NaN
2,2024,8,Masculino,2024,Pública,7.9522,0,ALFA (1° e 2° ano),0,10.002,...,7.046667,10.0,Ametista,NaN,NaN,RA-1277,NaN,NaN,NaN,NaN
3,2024,8,Masculino,2023,Pública,7.156367,0,Fase 1 (3° e 4° ano),-1,8.002,...,7.213333,5.0,Ametista,Topázio,NaN,RA-868,NaN,NaN,NaN,8.63895
4,2024,9,Masculino,2024,Pública,5.4442,0,Fase 1 (3° e 4° ano),-1,9.002,...,4.173333,5.0,Quartzo,NaN,NaN,RA-1278,NaN,NaN,NaN,NaN


In [ ]:
df_completo = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)

In [ ]:
df_completo.describe()

,Ano,Idade,Ano ingresso,Fase,Defasagem,IAA,IEG,IPS,IPP,IDA,Mat,Por,Ing,IPV,IAN,INDE - 1,INDE - 2
count,3030.00000,3030.000000,3030.000000,3030.000000,3030.000000,2865.000000,2954.000000,2859.000000,2852.000000,2852.000000,2846.000000,2845.000000,1091.000000,2852.000000,3030.000000,1290.000000,472.000000
mean,2023.09769,12.585809,2021.563696,2.557426,-0.642904,7.918225,7.945696,6.287129,7.171427,6.375964,6.161595,6.430668,6.289413,7.545476,7.179043,7.365852,7.368276
std,0.80995,3.300869,1.822171,2.255364,0.866382,2.626209,2.152281,1.792491,1.159301,1.956637,2.398341,2.138916,2.732636,1.084347,2.535266,0.876740,0.861821
min,2022.00000,7.000000,2016.000000,0.000000,-5.000000,0.000000,0.000000,2.500000,-0.074000,0.000000,0.000000,0.000000,0.000000,2.500000,2.500000,3.700000,3.031806
25%,2022.00000,10.000000,2021.000000,1.000000,-1.000000,7.900000,7.300000,5.020000,6.329500,5.100000,4.800000,5.200000,4.500000,6.984000,5.000000,6.804000,6.890881
50%,2023.00000,12.000000,2022.000000,2.000000,-1.000000,8.751000,8.600000,7.500000,7.500000,6.666667,6.300000,6.700000,6.700000,7.583000,5.000000,7.455500,7.475431
75%,2024.00000,15.000000,2023.000000,4.000000,0.000000,9.500000,9.400000,7.510000,7.968750,7.833333,8.000000,8.000000,8.500000,8.255000,10.000000,7.987185,7.981160
max,2024.00000,27.000000,2024.000000,9.000000,3.000000,10.002000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.010000,10.000000,9.442000,9.441522


In [ ]:
df_completo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Ano                    3030 non-null   int64  
 1   Idade                  3030 non-null   int64  
 2   Gênero                 3030 non-null   object 
 3   Ano ingresso           3030 non-null   int64  
 4   Instituição de ensino  3029 non-null   object 
 5   INDE                   2883 non-null   object 
 6   Fase                   3030 non-null   int64  
 7   Fase Ideal             3030 non-null   object 
 8   Defasagem              3030 non-null   int64  
 9   IAA                    2865 non-null   float64
 10  IEG                    2954 non-null   float64
 11  IPS                    2859 non-null   float64
 12  IPP                    2852 non-null   float64
 13  IDA                    2852 non-null   float64
 14  Mat                    2846 non-null   float64
 15  Por 

In [ ]:
df_completo['Fase Ideal'] = pd.to_numeric(df_completo['Fase Ideal'], errors='coerce')
df_completo['Fase Ideal'] = df_completo['Fase'] - df_completo['Defasagem']

In [ ]:
colunas_float = df_completo.select_dtypes(include=['float64']).columns
casas_decimais = 2  # Número de casas decimais desejadas

for coluna in colunas_float:
    df_completo[coluna] = df_completo[coluna].round(casas_decimais)

In [ ]:
df_completo.sample(5)

,Ano,Idade,Gênero,Ano ingresso,Instituição de ensino,INDE,Fase,Fase Ideal,Defasagem,IAA,...,IPV,IAN,Pedra,Pedra -1,Pedra -2,RA,Pedra -3,INDE - 1,Pedra -4,INDE - 2
2542,2024,14,Feminino,2024,Pública,9.089829,3,3,0,9.17,...,8.34,10.0,Topázio,NaN,NaN,RA-1533,NaN,NaN,NaN,NaN
134,2022,15,Menina,2019,Escola Pública,7.692,4,5,-1,8.30,...,8.58,5.0,Ametista,Ametista,Topázio,RA-135,NaN,NaN,NaN,NaN
2268,2024,11,Feminino,2023,Pública,6.643252,2,2,0,7.92,...,5.66,10.0,Agata,Ametista,NaN,RA-1015,NaN,7.54,NaN,NaN
346,2022,12,Menina,2019,Escola Pública,7.908,2,3,-1,9.00,...,8.00,5.0,Ametista,Ametista,Topázio,RA-347,NaN,NaN,NaN,NaN
804,2022,8,Menino,2021,Escola Pública,8.279,0,0,0,9.50,...,7.33,10.0,Topázio,Ametista,NaN,RA-805,NaN,NaN,NaN,NaN


In [ ]:
# 1. Tratando a coluna INDE: substitui vírgula por ponto e força para número (textos como 'INCLUIR' viram NaN)
df_completo['INDE'] = df_completo['INDE'].astype(str).str.replace(',', '.', regex=False)
df_completo['INDE'] = pd.to_numeric(df_completo['INDE'], errors='coerce').round(2)

In [ ]:
valores_unicos = df_completo['Instituição de ensino'].unique()
print(valores_unicos)

['Escola Pública' 'Rede Decisão' 'Escola JP II' 'Pública' 'Privada'
 'Privada - Programa de Apadrinhamento'
 'Privada - Programa de apadrinhamento' 'Concluiu o 3º EM'
 'Nenhuma das opções acima' 'Privada *Parcerias com Bolsa 100%'
 'Privada - Pagamento por *Empresa Parceira' nan
 'Bolsista Universitário *Formado (a)']


In [ ]:
# 1. Dicionário de mapeamento para agrupar as variações
mapeamento_instituicao = {
    'Escola Pública': 'Escola Pública',
    'Pública': 'Escola Pública',

    'Rede Decisão': 'Escola Privada',
    'Escola JP II': 'Escola Privada',
    'Privada': 'Escola Privada',
    'Privada - Programa de Apadrinhamento': 'Escola Privada',
    'Privada - Programa de apadrinhamento': 'Escola Privada',
    'Privada *Parcerias com Bolsa 100%': 'Escola Privada',
    'Privada - Pagamento por *Empresa Parceira': 'Escola Privada',

    'Concluiu o 3º EM': 'Formado/Universitário',
    'Bolsista Universitário *Formado (a)': 'Formado/Universitário',

    'Nenhuma das opções acima': 'Outros'
}

# 2. Aplicando a substituição na base completa
df_completo['Instituição de ensino'] = df_completo['Instituição de ensino'].replace(mapeamento_instituicao)

# 3. Tratando os valores nulos (NaN) que restaram
df_completo['Instituição de ensino'] = df_completo['Instituição de ensino'].fillna('Outros')

# 4. Verificando o resultado final
print("Distribuição após a limpeza:")
print(df_completo['Instituição de ensino'].value_counts())

Distribuição após a limpeza:
Instituição de ensino
Escola Pública           2474
Escola Privada            526
Formado/Universitário      27
Outros                      3
Name: count, dtype: int64


In [ ]:
valores_unicos = df_completo['Pedra'].unique()
print(valores_unicos)

['Quartzo' 'Ametista' 'Ágata' 'Topázio' 'Agata' nan 'INCLUIR']


In [ ]:
# Substitui 'Agata' (sem acento) por 'Ágata' (com acento) na coluna 'Pedra'
df_completo['Pedra'] = df_completo['Pedra'].replace({'Agata': 'Ágata'})

print(df_completo['Pedra'].unique())
# Saída esperada: ['Quartzo' 'Ametista' 'Ágata' 'Topázio']

['Quartzo' 'Ametista' 'Ágata' 'Topázio' nan 'INCLUIR']


In [ ]:
df_completo = df_completo.dropna(subset=['INDE'])
print(f"Tamanho após remover alunos sem INDE: {df_completo.shape}")

Tamanho após remover alunos sem INDE: (2845, 27)


In [ ]:
valores_unicos = df_completo['Gênero'].unique()
print(valores_unicos)

['Menina' 'Menino' 'Feminino' 'Masculino']


In [ ]:
mapeamento_genero = {
    'Menina': 'Feminino',
    'Menino': 'Masculino',
    'Feminino': 'Feminino',
    'Masculino': 'Masculino'
}

# Aplicando a substituição na base completa
df_completo['Gênero'] = df_completo['Gênero'].replace(mapeamento_genero)

In [ ]:
df_completo['Ano de Nascimento'] = df_completo['Ano'] - df_completo['Idade']
df_completo = df_completo.drop(columns=['Idade'])

In [ ]:
df_completo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2845 entries, 0 to 2927
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Ano                    2845 non-null   int64  
 1   Gênero                 2845 non-null   object 
 2   Ano ingresso           2845 non-null   int64  
 3   Instituição de ensino  2845 non-null   object 
 4   INDE                   2845 non-null   float64
 5   Fase                   2845 non-null   int64  
 6   Fase Ideal             2845 non-null   int64  
 7   Defasagem              2845 non-null   int64  
 8   IAA                    2845 non-null   float64
 9   IEG                    2845 non-null   float64
 10  IPS                    2845 non-null   float64
 11  IPP                    2845 non-null   float64
 12  IDA                    2845 non-null   float64
 13  Mat                    2839 non-null   float64
 14  Por                    2838 non-null   float64
 15  Ing      

In [ ]:
ordem_desejada_final = [
    'Ano',
    'RA',
    'Ano de Nascimento',
    'Gênero',
    'Ano ingresso',
    'Instituição de ensino',
    'Fase',
    'Fase Ideal',
    'Defasagem',
    'INDE',
    'IAN',
    'IDA',
    'Mat',
    'Por',
    'Ing',
    'IEG',
    'IAA',
    'IPS',
    'IPP',
    'IPV',
    'Pedra',
    'INDE - 1',
    'INDE - 2',
    'Pedra -1',
    'Pedra -2',
    'Pedra -3',
    'Pedra -4'
]

# Reapplying the reorder function with the new, more complete order
df_completo = reorder_columns_dynamically(df_completo, ordem_desejada_final)

print("DataFrame com colunas reordenadas:")
df_completo.info()
df_completo.head()

DataFrame com colunas reordenadas:
<class 'pandas.core.frame.DataFrame'>
Index: 2845 entries, 0 to 2927
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Ano                    2845 non-null   int64  
 1   RA                     2845 non-null   object 
 2   Ano de Nascimento      2845 non-null   int64  
 3   Gênero                 2845 non-null   object 
 4   Ano ingresso           2845 non-null   int64  
 5   Instituição de ensino  2845 non-null   object 
 6   Fase                   2845 non-null   int64  
 7   Fase Ideal             2845 non-null   int64  
 8   Defasagem              2845 non-null   int64  
 9   INDE                   2845 non-null   float64
 10  IAN                    2845 non-null   float64
 11  IDA                    2845 non-null   float64
 12  Mat                    2839 non-null   float64
 13  Por                    2838 non-null   float64
 14  Ing                    108

,Ano,RA,Ano de Nascimento,Gênero,Ano ingresso,Instituição de ensino,Fase,Fase Ideal,Defasagem,INDE,...,IPS,IPP,IPV,Pedra,INDE - 1,INDE - 2,Pedra -1,Pedra -2,Pedra -3,Pedra -4
0,2022,RA-1,2003,Feminino,2016,Escola Pública,7,8,-1,5.78,...,5.6,8.17,7.28,Quartzo,NaN,NaN,Ametista,Ametista,NaN,NaN
1,2022,RA-2,2005,Feminino,2017,Escola Privada,7,7,0,7.06,...,6.3,7.89,6.78,Ametista,NaN,NaN,Ametista,Ametista,NaN,NaN
2,2022,RA-3,2005,Feminino,2016,Escola Privada,7,7,0,6.59,...,5.6,8.20,7.56,Ágata,NaN,NaN,Ametista,Ametista,NaN,NaN
3,2022,RA-4,2005,Masculino,2017,Escola Privada,7,7,0,5.95,...,5.6,5.55,5.28,Quartzo,NaN,NaN,Ametista,Ametista,NaN,NaN
4,2022,RA-5,2005,Feminino,2016,Escola Privada,7,7,0,7.43,...,5.6,8.39,7.39,Ametista,NaN,NaN,Ametista,Ametista,NaN,NaN


In [ ]:
df_completo['RA'] = df_completo['RA'].astype(str).str.replace('RA-', '', regex=False)
df_completo['RA'] = df_completo['RA'].astype(int)
df_completo = df_completo.sort_values(by=['Ano', 'RA']).reset_index(drop=True)
print("DataFrame ordenado por 'Ano' e 'RA' (com RA sem prefixo e como inteiro):")
df_completo.head()

DataFrame ordenado por 'Ano' e 'RA' (com RA sem prefixo e como inteiro):


,Ano,RA,Ano de Nascimento,Gênero,Ano ingresso,Instituição de ensino,Fase,Fase Ideal,Defasagem,INDE,...,IPS,IPP,IPV,Pedra,INDE - 1,INDE - 2,Pedra -1,Pedra -2,Pedra -3,Pedra -4
0,2022,1,2003,Feminino,2016,Escola Pública,7,8,-1,5.78,...,5.6,8.17,7.28,Quartzo,NaN,NaN,Ametista,Ametista,NaN,NaN
1,2022,2,2005,Feminino,2017,Escola Privada,7,7,0,7.06,...,6.3,7.89,6.78,Ametista,NaN,NaN,Ametista,Ametista,NaN,NaN
2,2022,3,2005,Feminino,2016,Escola Privada,7,7,0,6.59,...,5.6,8.20,7.56,Ágata,NaN,NaN,Ametista,Ametista,NaN,NaN
3,2022,4,2005,Masculino,2017,Escola Privada,7,7,0,5.95,...,5.6,5.55,5.28,Quartzo,NaN,NaN,Ametista,Ametista,NaN,NaN
4,2022,5,2005,Feminino,2016,Escola Privada,7,7,0,7.43,...,5.6,8.39,7.39,Ametista,NaN,NaN,Ametista,Ametista,NaN,NaN


In [ ]:
df_completo.describe()

,Ano,RA,Ano de Nascimento,Ano ingresso,Fase,Fase Ideal,Defasagem,INDE,IAN,IDA,Mat,Por,Ing,IEG,IAA,IPS,IPP,IPV,INDE - 1,INDE - 2
count,2845.000000,2845.000000,2845.000000,2845.000000,2845.000000,2845.000000,2845.000000,2845.000000,2845.000000,2845.000000,2839.000000,2838.000000,1084.000000,2845.000000,2845.000000,2845.000000,2845.000000,2845.000000,1248.000000,445.000000
mean,2023.068190,707.295958,2010.956766,2021.610545,2.205272,2.904745,-0.699473,7.269902,7.010545,6.376812,6.162416,6.431483,6.290452,8.229736,7.926271,6.297842,7.170886,7.546039,7.363093,7.376787
std,0.817523,421.094490,2.861899,1.835201,1.822477,1.896513,0.849847,0.991702,2.510504,1.957513,2.400191,2.140832,2.734547,1.568507,2.617448,1.784455,1.159355,1.085591,0.868295,0.847669
min,2022.000000,1.000000,2001.000000,2016.000000,0.000000,0.000000,-5.000000,3.030000,2.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.500000,-0.070000,2.500000,3.700000,3.030000
25%,2022.000000,369.000000,2009.000000,2021.000000,1.000000,2.000000,-1.000000,6.680000,5.000000,5.100000,4.800000,5.200000,4.500000,7.500000,7.900000,5.020000,6.330000,6.970000,6.807500,6.920000
50%,2023.000000,672.000000,2011.000000,2022.000000,2.000000,3.000000,-1.000000,7.390000,5.000000,6.670000,6.300000,6.700000,6.700000,8.650000,8.750000,7.500000,7.500000,7.580000,7.450000,7.480000
75%,2024.000000,1009.000000,2013.000000,2023.000000,3.000000,4.000000,0.000000,7.990000,10.000000,7.830000,8.000000,8.000000,8.500000,9.400000,9.500000,7.510000,7.970000,8.260000,7.982500,7.990000
max,2024.000000,1632.000000,2017.000000,2024.000000,7.000000,8.000000,3.000000,9.530000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.010000,9.440000,9.440000


In [ ]:
df_completo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2845 entries, 0 to 2844
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Ano                    2845 non-null   int64  
 1   RA                     2845 non-null   int64  
 2   Ano de Nascimento      2845 non-null   int64  
 3   Gênero                 2845 non-null   object 
 4   Ano ingresso           2845 non-null   int64  
 5   Instituição de ensino  2845 non-null   object 
 6   Fase                   2845 non-null   int64  
 7   Fase Ideal             2845 non-null   int64  
 8   Defasagem              2845 non-null   int64  
 9   INDE                   2845 non-null   float64
 10  IAN                    2845 non-null   float64
 11  IDA                    2845 non-null   float64
 12  Mat                    2839 non-null   float64
 13  Por                    2838 non-null   float64
 14  Ing                    1084 non-null   float64
 15  IEG 

In [ ]:
# # 1. Carregar a base de dados original
# df_original = df_completo

# # 2. Ordenar e filtrar apenas alunos presentes nos 3 anos para o cálculo
# df_ordenado = df_original.sort_values(by=['RA', 'Ano'])
# alunos_3_anos = df_ordenado.groupby('RA')['Ano'].nunique()
# ra_validos = alunos_3_anos[alunos_3_anos == 3].index
# df_filtrado = df_ordenado[df_ordenado['RA'].isin(ra_validos)].copy()

# # 3. Pivotar a Defasagem
# df_defasagem = df_filtrado.pivot(index='RA', columns='Ano', values='Defasagem')

# # 4. Criar a tag (com 2024 como NaN/vazio)
# df_tag_antecipada = pd.DataFrame(index=df_defasagem.index)
# df_tag_antecipada[2022] = (df_defasagem[2023] < df_defasagem[2022]).astype(int)
# df_tag_antecipada[2023] = (df_defasagem[2024] < df_defasagem[2023]).astype(int)
# df_tag_antecipada[2024] = np.nan # Define como vazio (NaN)

# # 5. Despivotar (Melt) para o formato longo
# df_tag_longo = df_tag_antecipada.reset_index().melt(
#     id_vars='RA',
#     var_name='Ano',
#     value_name='Aumento_Df_Ano_Seguinte'
# )
# df_tag_longo['Ano'] = df_tag_longo['Ano'].astype(int)

# # 6. Unir (Merge) de volta ao DataFrame original inteiro
# df_resultado_final = df_original.merge(df_tag_longo, on=['RA', 'Ano'], how='left')

# # 7. Salvar o arquivo
# df_resultado_final.to_csv('df_passos_magicos_com_tag.csv', index=False)

In [ ]:
# 1. Carregar a base de dados original
df_original = df_completo

# 2. Ordenar de forma cronológica por Aluno e Ano
df_ordenado = df_original.sort_values(by=['RA', 'Ano']).copy()

# 3. Trazer os dados do ano seguinte para a linha do ano atual usando shift(-1)
df_ordenado['Proximo_Ano'] = df_ordenado.groupby('RA')['Ano'].shift(-1)
df_ordenado['Proxima_Defasagem'] = df_ordenado.groupby('RA')['Defasagem'].shift(-1)

# 4. Validar se o próximo registro é realmente o ano seguinte consecutivo (Ano + 1)
# Isso impede que o modelo compare 2022 com 2024 se o aluno sumiu em 2023.
transicao_valida = (df_ordenado['Proximo_Ano'] == df_ordenado['Ano'] + 1)

# 5. Criar a Coluna Alvo (Target) baseada na transição consecutiva
df_ordenado['Aumento_Df_Ano_Seguinte'] = np.nan
df_ordenado.loc[transicao_valida, 'Aumento_Df_Ano_Seguinte'] = (
    df_ordenado.loc[transicao_valida, 'Proxima_Defasagem'] < df_ordenado.loc[transicao_valida, 'Defasagem']
).astype(int)

# 6. Limpar colunas auxiliares que usamos no cálculo
df_resultado_final = df_ordenado.drop(columns=['Proximo_Ano', 'Proxima_Defasagem'])

# 7. Salvar o arquivo final com muito mais dados para o ML
df_resultado_final.to_csv('df_passos_magicos_com_tag_maxima.csv', index=False)

print("Nova distribuição da Variável Alvo (sem nulos):")
print(df_resultado_final['Aumento_Df_Ano_Seguinte'].value_counts())

Nova distribuição da Variável Alvo (sem nulos):
Aumento_Df_Ano_Seguinte
0.0    1022
1.0     226
Name: count, dtype: int64


In [ ]:
df_resultado_final.head()

,Ano,RA,Ano de Nascimento,Gênero,Ano ingresso,Instituição de ensino,Fase,Fase Ideal,Defasagem,INDE,...,IPP,IPV,Pedra,INDE - 1,INDE - 2,Pedra -1,Pedra -2,Pedra -3,Pedra -4,Aumento_Df_Ano_Seguinte
0,2022,1,2003,Feminino,2016,Escola Pública,7,8,-1,5.78,...,8.17,7.28,Quartzo,NaN,NaN,Ametista,Ametista,NaN,NaN,NaN
1,2022,2,2005,Feminino,2017,Escola Privada,7,7,0,7.06,...,7.89,6.78,Ametista,NaN,NaN,Ametista,Ametista,NaN,NaN,NaN
2,2022,3,2005,Feminino,2016,Escola Privada,7,7,0,6.59,...,8.20,7.56,Ágata,NaN,NaN,Ametista,Ametista,NaN,NaN,NaN
3,2022,4,2005,Masculino,2017,Escola Privada,7,7,0,5.95,...,5.55,5.28,Quartzo,NaN,NaN,Ametista,Ametista,NaN,NaN,NaN
4,2022,5,2005,Feminino,2016,Escola Privada,7,7,0,7.43,...,8.39,7.39,Ametista,NaN,NaN,Ametista,Ametista,NaN,NaN,NaN


In [ ]:
df_resultado_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2845 entries, 0 to 2844
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Ano                      2845 non-null   int64  
 1   RA                       2845 non-null   int64  
 2   Ano de Nascimento        2845 non-null   int64  
 3   Gênero                   2845 non-null   object 
 4   Ano ingresso             2845 non-null   int64  
 5   Instituição de ensino    2845 non-null   object 
 6   Fase                     2845 non-null   int64  
 7   Fase Ideal               2845 non-null   int64  
 8   Defasagem                2845 non-null   int64  
 9   INDE                     2845 non-null   float64
 10  IAN                      2845 non-null   float64
 11  IDA                      2845 non-null   float64
 12  Mat                      2839 non-null   float64
 13  Por                      2838 non-null   float64
 14  Ing                      1084

In [ ]:
# Lista de todas as colunas qualitativas, nominais ou de rankings que não serão usadas no modelo
colunas_para_remover = [
    'INDE - 1', 'INDE - 2', 'Pedra -1', 'Pedra -2', 'Pedra -3', 'Pedra -4'
]

In [ ]:
df_resultado_final = df_resultado_final.drop(columns=colunas_para_remover, errors='ignore')

In [ ]:
df_resultado_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2845 entries, 0 to 2844
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Ano                      2845 non-null   int64  
 1   RA                       2845 non-null   int64  
 2   Ano de Nascimento        2845 non-null   int64  
 3   Gênero                   2845 non-null   object 
 4   Ano ingresso             2845 non-null   int64  
 5   Instituição de ensino    2845 non-null   object 
 6   Fase                     2845 non-null   int64  
 7   Fase Ideal               2845 non-null   int64  
 8   Defasagem                2845 non-null   int64  
 9   INDE                     2845 non-null   float64
 10  IAN                      2845 non-null   float64
 11  IDA                      2845 non-null   float64
 12  Mat                      2839 non-null   float64
 13  Por                      2838 non-null   float64
 14  Ing                      1084

In [ ]:
from google.colab import files

# 2. Salvar o DataFrame como CSV no ambiente do Colab
#df_completo.to_csv('df_passos_magicos.csv', index=False)
df_resultado_final.to_csv('df_passos_magicos.csv', index=False)

# 3. Fazer o download do arquivo para o seu computador
files.download('df_passos_magicos.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>